In [2]:
"""
Model Comparison Script
Compares RL_ddm_biased vs AI_ddm (mtype=3, linear)
"""

import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# ==============================
# FILE PATHS
# ==============================
rl_path = 'magic_carpet_2020/fitted_parameters_MRL_ddm_biased_DRMlinear.csv'
df_rl = pd.read_csv(rl_path)

ai_path = 'magic_carpet_2020/fitted_parameters_MAI_ddm3_DRMlinear_n_starts3.csv'
df_ai = pd.read_csv(ai_path)

# Merge on ParticipantID
df = pd.merge(df_rl, df_ai, on="ParticipantID", suffixes=("_RL", "_AI"))

print(f"Merged participants: {len(df)}\n")

# ==============================
# BASIC MODEL FIT COMPARISON
# ==============================

print("===== MODEL FIT COMPARISON =====")

mean_nll_rl = df["NLL_RL"].mean()
mean_nll_ai = df["NLL_AI"].mean()

print(f"Mean NLL RL_ddm_biased: {mean_nll_rl:.2f}")
print(f"Mean NLL AI_ddm:        {mean_nll_ai:.2f}")

# Number of free parameters
k_rl = 10
k_ai = 10

# Approximate trial count
n_trials = 200  # adjust if needed

# AIC
df["AIC_RL"] = 2 * k_rl + 2 * df["NLL_RL"]
df["AIC_AI"] = 2 * k_ai + 2 * df["NLL_AI"]

# BIC
df["BIC_RL"] = k_rl * np.log(n_trials) + 2 * df["NLL_RL"]
df["BIC_AI"] = k_ai * np.log(n_trials) + 2 * df["NLL_AI"]

print("\nMean AIC RL:", df["AIC_RL"].mean())
print("Mean AIC AI:", df["AIC_AI"].mean())

print("\nMean BIC RL:", df["BIC_RL"].mean())
print("Mean BIC AI:", df["BIC_AI"].mean())

# ==============================
# PARAMETER CORRELATIONS
# ==============================

print("\n===== CROSS-MODEL PARAMETER CORRELATIONS =====")

param_pairs = [
    ("Fitted_a_bs_RL", "Fitted_a_bs_AI"),
    ("Fitted_ndt_RL", "Fitted_ndt_AI"),
    ("Fitted_v_stage_0_RL", "Fitted_v_stage_0_AI"),
    ("Fitted_v_stage_1_RL", "Fitted_v_stage_1_AI"),
]

for p1, p2 in param_pairs:
    r, pval = stats.pearsonr(df[p1], df[p2])
    print(f"{p1} vs {p2} → r = {r:.3f}, p = {pval:.4f}")

# ==============================
# RL vs AI Learning Comparison
# ==============================

print("\n===== LEARNING PARAMETER COMPARISON =====")

if "Fitted_lr1_RL" in df.columns and "Fitted_lr_AI" in df.columns:
    r, pval = stats.pearsonr(df["Fitted_lr1_RL"], df["Fitted_lr_AI"])
    print(f"lr1 (RL) vs lr (AI) → r = {r:.3f}, p = {pval:.4f}")



Merged participants: 23

===== MODEL FIT COMPARISON =====
Mean NLL RL_ddm_biased: 0.59
Mean NLL AI_ddm:        0.54

Mean AIC RL: 21.181252128045575
Mean AIC AI: 21.078109011280446

Mean BIC RL: 54.16442579352595
Mean BIC AI: 54.06128267676081

===== CROSS-MODEL PARAMETER CORRELATIONS =====
Fitted_a_bs_RL vs Fitted_a_bs_AI → r = 0.979, p = 0.0000
Fitted_ndt_RL vs Fitted_ndt_AI → r = 1.000, p = 0.0000
Fitted_v_stage_0_RL vs Fitted_v_stage_0_AI → r = 0.531, p = 0.0092
Fitted_v_stage_1_RL vs Fitted_v_stage_1_AI → r = 0.140, p = 0.5232

===== LEARNING PARAMETER COMPARISON =====
